# J-Space Experiment — Phases 1–4 (Local Linux)

End-to-end launcher for a local Linux machine with CUDA. Each phase runs through the same CLI used from the terminal. Results are saved under one run root inside this repository.

Default configuration: **Qwen 3.5 9B smoke** (`configs/phase1_qwen35_9b_smoke.yaml`).

On multi-GPU machines, set `PHYSICAL_GPU_INDEX` in section 4 (for example `0`, `1`, `2`, or `3` on a 4x L4 node). The notebook remaps that card to logical `cuda:0` with `CUDA_VISIBLE_DEVICES`. Section 5b can launch independent jobs in parallel across multiple GPUs.

The setup cell tries a project `.venv` first, then automatically falls back to the Jupyter kernel with `pip install --user` on restricted hubs such as TLJH. Pipeline commands always run through `python -m jspace_research...`, so they do not depend on `jspace-phase*` scripts being on `PATH`.

On JupyterHub, you can also set `USE_PROJECT_VENV = False` at the top of section 1 to skip `.venv` entirely.

Credentials can be entered manually in section 2 if they are not already in the shell environment.

## 1. Install dependencies and verify pinned checkouts

In [ ]:
import os
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

AGENTDOJO_REVISION = '089ed468cf3ed0322acc66b0211f26d9d90dbf60'
INJECAGENT_REVISION = 'f19c9f2c79a41046eb13c03c51a24c567a8ffa07'
PIP_EXTRAS = 'phase4,notebook'
USE_PROJECT_VENV = True  # set False on JupyterHub if .venv setup keeps failing
CLI_MODULES = {
    'jspace-phase1': 'jspace_research.phase1.cli',
    'jspace-phase2': 'jspace_research.phase2.cli',
    'jspace-phase3': 'jspace_research.phase3.cli',
    'jspace-phase4': 'jspace_research.phase4.cli',
}


def runtime_command_env(venv_dir: Path | None, physical_gpu_index: int | None = None) -> dict[str, str]:
    env = os.environ.copy()
    env['PIP_USER'] = '0' if venv_dir is not None else '1'
    env.pop('PYTHONHOME', None)
    if venv_dir is not None:
        venv_bin = venv_dir / 'bin'
        env['VIRTUAL_ENV'] = str(venv_dir)
        env['PATH'] = f'{venv_bin}{os.pathsep}{env.get("PATH", "")}'
        env['PYTHONNOUSERSITE'] = '1'
    if physical_gpu_index is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(physical_gpu_index)
    return env


def resolve_cli_command(command: list[str], runtime_python: Path) -> list[str]:
    if command and command[0] in CLI_MODULES:
        return [str(runtime_python), '-m', CLI_MODULES[command[0]], *command[1:]]
    return command


def _venv_python_path(venv_dir: Path) -> Path:
    bin_dir = venv_dir / 'bin'
    for name in (
        'python',
        'python3',
        f'python{sys.version_info.major}',
        f'python{sys.version_info.major}.{sys.version_info.minor}',
    ):
        candidate = bin_dir / name
        if candidate.exists():
            return candidate
    return bin_dir / 'python'


def _venv_prefix_matches(venv_dir: Path, venv_python: Path) -> bool:
    result = subprocess.run(
        [str(venv_python), '-c', 'import sys; print(sys.prefix)'],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        return False
    prefix = Path(result.stdout.strip())
    try:
        return os.path.samefile(prefix, venv_dir)
    except OSError:
        return prefix.resolve() == venv_dir.resolve()


def _venv_is_valid(venv_dir: Path) -> bool:
    venv_python = _venv_python_path(venv_dir)
    return (venv_dir / 'pyvenv.cfg').is_file() and venv_python.exists() and _venv_prefix_matches(
        venv_dir, venv_python
    )


def _run_pip_install(
    python_executable: Path,
    repo_root: Path,
    *,
    env: dict[str, str],
    editable: bool,
    use_user_site: bool,
) -> None:
    command = [str(python_executable), '-Im', 'pip', 'install']
    if use_user_site:
        command.append('--user')
    else:
        command.append('--no-user')
    if editable:
        command.extend(['-e', f'{repo_root}[{PIP_EXTRAS}]'])
    else:
        command.append(f'{repo_root}[{PIP_EXTRAS}]')
    result = subprocess.run(
        command,
        cwd=repo_root,
        env=env,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    if result.returncode != 0:
        raise RuntimeError(
            'pip install failed with command: '
            + ' '.join(command)
            + f'\nexit code: {result.returncode}'
        )


def _import_check(python_executable: Path, env: dict[str, str], repo_root: Path) -> bool:
    result = subprocess.run(
        [
            str(python_executable),
            '-c',
            (
                'import jspace_research, pandas, torch; '
                'from jspace_research.phase1.cli import main'
            ),
        ],
        cwd=repo_root,
        env=env,
        capture_output=True,
        text=True,
        check=False,
    )
    return result.returncode == 0


def _create_project_venv(repo_root: Path) -> Path:
    venv_dir = (repo_root / '.venv').resolve()
    if venv_dir.exists():
        shutil.rmtree(venv_dir)
    subprocess.run(
        [sys.executable, '-m', 'venv', '--clear', str(venv_dir)],
        cwd=repo_root,
        check=True,
    )
    if not _venv_is_valid(venv_dir):
        raise RuntimeError(f'Created .venv at {venv_dir}, but sys.prefix does not point there.')
    return venv_dir


def _ensure_project_venv(repo_root: Path) -> tuple[Path, Path, Path]:
    venv_dir = (repo_root / '.venv').resolve()
    if not _venv_is_valid(venv_dir):
        venv_dir = _create_project_venv(repo_root)
    venv_python = _venv_python_path(venv_dir)
    install_env = runtime_command_env(venv_dir)
    if not _import_check(venv_python, install_env, repo_root):
        for editable in (True, False):
            try:
                _run_pip_install(
                    venv_python,
                    repo_root,
                    env=install_env,
                    editable=editable,
                    use_user_site=False,
                )
                break
            except RuntimeError:
                if editable:
                    warnings.warn(
                        'Editable install into .venv failed; retrying non-editable install.',
                        stacklevel=2,
                    )
                    continue
                raise
    if not _import_check(venv_python, install_env, repo_root):
        raise RuntimeError('Project virtualenv was created, but jspace_research is still not importable.')
    return venv_python, venv_dir / 'bin', venv_dir


def _ensure_kernel_runtime(repo_root: Path) -> tuple[Path, None, None]:
    runtime_python = Path(sys.executable).resolve()
    install_env = runtime_command_env(None)
    if not _import_check(runtime_python, install_env, repo_root):
        for editable in (True, False):
            try:
                _run_pip_install(
                    runtime_python,
                    repo_root,
                    env=install_env,
                    editable=editable,
                    use_user_site=True,
                )
                break
            except RuntimeError:
                if editable:
                    warnings.warn(
                        'Editable --user install failed; retrying non-editable --user install.',
                        stacklevel=2,
                    )
                    continue
                raise
    if not _import_check(runtime_python, install_env, repo_root):
        raise RuntimeError('Kernel Python still cannot import jspace_research after pip install --user.')
    return runtime_python, None, None


def ensure_runtime(repo_root: Path) -> tuple[Path, Path | None, Path | None]:
    if USE_PROJECT_VENV:
        try:
            runtime_python, runtime_bin, runtime_dir = _ensure_project_venv(repo_root)
            print('Runtime mode: project .venv')
            return runtime_python, runtime_bin, runtime_dir
        except Exception as exc:
            warnings.warn(
                f'Project .venv setup failed; falling back to Jupyter kernel Python with pip --user. {exc}',
                stacklevel=2,
            )
    runtime_python, runtime_bin, runtime_dir = _ensure_kernel_runtime(repo_root)
    print('Runtime mode: Jupyter kernel Python (--user install)')
    return runtime_python, runtime_bin, runtime_dir


def verify_notebook_kernel(runtime_python: Path, runtime_dir: Path | None) -> None:
    current_python = Path(sys.executable).resolve()
    if runtime_dir is not None and current_python != runtime_python.resolve():
        warnings.warn(
            'Notebook kernel is not the project virtualenv. '
            'Pipeline commands still run through the selected runtime Python.',
            stacklevel=2,
        )


def verify_project_environment(
    repo_root: Path,
    runtime_python: Path,
    runtime_dir: Path | None,
) -> None:
    env = runtime_command_env(runtime_dir)
    subprocess.run(
        resolve_cli_command(['jspace-phase1', '--help'], runtime_python),
        cwd=repo_root,
        env=env,
        check=True,
        stdout=subprocess.DEVNULL,
    )


cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / 'pyproject.toml').is_file() else cwd.parent
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Run this notebook from the repository root or notebooks/ directory.')

RUNTIME_PYTHON, RUNTIME_BIN, RUNTIME_DIR = ensure_runtime(REPO_ROOT)
VENV_PYTHON = RUNTIME_PYTHON
VENV_BIN = RUNTIME_BIN or Path(sys.executable).resolve().parent
VENV_DIR = RUNTIME_DIR
verify_notebook_kernel(RUNTIME_PYTHON, RUNTIME_DIR)
verify_project_environment(REPO_ROOT, RUNTIME_PYTHON, RUNTIME_DIR)
print('Runtime python:', RUNTIME_PYTHON)

BENCHMARKS_ROOT = Path(
    os.environ.get('JSPACE_BENCHMARKS_ROOT', REPO_ROOT.parent / 'jspace-benchmarks')
).expanduser().resolve()
BIPIA_CHECKOUT = REPO_ROOT / 'BIPIA'
AGENTDOJO_CHECKOUT = BENCHMARKS_ROOT / 'agentdojo'
INJECAGENT_CHECKOUT = BENCHMARKS_ROOT / 'InjecAgent'

subprocess.run(['git', 'submodule', 'update', '--init', 'BIPIA'], cwd=REPO_ROOT, check=True)

BENCHMARKS_ROOT.mkdir(parents=True, exist_ok=True)
if not AGENTDOJO_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/ethz-spylab/agentdojo.git', str(AGENTDOJO_CHECKOUT)],
        check=True,
    )
subprocess.run(['git', '-C', str(AGENTDOJO_CHECKOUT), 'checkout', AGENTDOJO_REVISION], check=True)
if not INJECAGENT_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/uiuc-kang-lab/InjecAgent.git', str(INJECAGENT_CHECKOUT)],
        check=True,
    )
subprocess.run(['git', '-C', str(INJECAGENT_CHECKOUT), 'checkout', INJECAGENT_REVISION], check=True)

print('Repository root:', REPO_ROOT)
print('Benchmarks root:', BENCHMARKS_ROOT)
print('Research revision:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('BIPIA revision:', subprocess.check_output(['git', '-C', str(BIPIA_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('AgentDojo revision:', subprocess.check_output(['git', '-C', str(AGENTDOJO_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('InjecAgent revision:', subprocess.check_output(['git', '-C', str(INJECAGENT_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())

## 2. Enter credentials manually (optional)

Paste tokens here when they are not already available in the shell environment. Leave a field empty to keep using the existing environment variable instead.

In [ ]:
import os

# Paste tokens here if needed. Leave blank to use the shell environment instead.
MANUAL_HF_TOKEN = ''
MANUAL_OPENROUTER_API_KEY = ''


def apply_manual_credentials() -> None:
    if MANUAL_HF_TOKEN.strip():
        os.environ['HF_TOKEN'] = MANUAL_HF_TOKEN.strip()
    if MANUAL_OPENROUTER_API_KEY.strip():
        os.environ['OPENROUTER_API_KEY'] = MANUAL_OPENROUTER_API_KEY.strip()


apply_manual_credentials()

print('HF token set:', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')))
print('OpenRouter key set:', bool(os.environ.get('OPENROUTER_API_KEY')))

## 3. Authenticate

In [ ]:
import os

from huggingface_hub import login

apply_manual_credentials()

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face token loaded.')
elif MANUAL_HF_TOKEN.strip():
    raise RuntimeError('MANUAL_HF_TOKEN was set but HF_TOKEN is still missing after apply_manual_credentials().')
else:
    login(add_to_git_credential=False)
    print('Hugging Face login complete.')

if not os.environ.get('OPENROUTER_API_KEY'):
    raise RuntimeError(
        'OpenRouter credential is missing. Set OPENROUTER_API_KEY in the shell environment '
        'or paste it into MANUAL_OPENROUTER_API_KEY in section 2, then rerun sections 2 and 3.'
    )
print('OpenRouter judge credential is set.')

## 4. Configure run directory and GPU selection

On a multi-GPU machine (for example 4x NVIDIA L4), set `PHYSICAL_GPU_INDEX` to the card this notebook should use. The pipeline still runs on logical `cuda:0`; the notebook remaps that to your chosen physical GPU with `CUDA_VISIBLE_DEVICES`.

To run several independent jobs in parallel, open one notebook per GPU or populate `PARALLEL_JOBS` in the optional launcher cell below.

In [ ]:
import threading

RUN_MODE = 'smoke'  # use 'full' only after smoke succeeds
MODEL_KEY = 'qwen35_9b'  # default local experiment model
PHYSICAL_GPU_INDEX = 0  # physical GPU id on this machine, e.g. 0-3 on a 4x L4 node
CONFIG_NAME = f'phase1_{MODEL_KEY}_{RUN_MODE}'
RUN_NAME = f'jspace-{MODEL_KEY}-{RUN_MODE}-gpu{PHYSICAL_GPU_INDEX}'

RUN_ROOT = REPO_ROOT / 'artifacts' / RUN_NAME
PHASE1_DIR = RUN_ROOT / 'phase1'
PHASE2_DIR = RUN_ROOT / 'phase2'
PHASE3_DIR = RUN_ROOT / 'phase3'
PHASE4_DIR = RUN_ROOT / 'phase4'
BIPIA_ROOT = BIPIA_CHECKOUT / 'benchmark'
CONFIG_PATH = REPO_ROOT / 'configs' / f'{CONFIG_NAME}.yaml'
WEBQA_TRAIN_PATH = None  # required for full mode
SUMMARIZATION_TRAIN_PATH = None  # required for full mode

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Configuration not found: {CONFIG_PATH}')


def pipeline_env(physical_gpu_index: int | None = None) -> dict[str, str]:
    return runtime_command_env(RUNTIME_DIR, physical_gpu_index)


def list_physical_gpus() -> None:
    result = subprocess.run(
        [
            str(RUNTIME_PYTHON),
            '-c',
            (
                'import torch; '
                'count = torch.cuda.device_count(); '
                'print(f"Visible GPU count: {count}"); '
                '[print(f"GPU {i}: {torch.cuda.get_device_name(i)}") for i in range(count)]'
            ),
        ],
        cwd=REPO_ROOT,
        env=pipeline_env(),
        check=True,
        capture_output=True,
        text=True,
    )
    print(result.stdout, end='')


def validate_gpu_index(physical_gpu_index: int) -> None:
    result = subprocess.run(
        [
            str(RUNTIME_PYTHON),
            '-c',
            (
                'import os, torch; '
                'index = int(os.environ["CUDA_VISIBLE_DEVICES"]); '
                'assert torch.cuda.is_available(), "CUDA is not available"; '
                'assert torch.cuda.device_count() == 1, "Expected one visible GPU"; '
                'print("Selected physical GPU:", index); '
                'print("Logical device:", torch.cuda.get_device_name(0))'
            ),
        ],
        cwd=REPO_ROOT,
        env=pipeline_env(physical_gpu_index),
        check=False,
        capture_output=True,
        text=True,
    )
    print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    if result.returncode != 0:
        raise RuntimeError(f'Physical GPU {physical_gpu_index} is not available.')


def run_command(command, label, *, physical_gpu_index: int | None = None):
    gpu_index = PHYSICAL_GPU_INDEX if physical_gpu_index is None else physical_gpu_index
    command = resolve_cli_command(command, RUNTIME_PYTHON)
    print(f'Running on physical GPU {gpu_index}:', ' '.join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=pipeline_env(gpu_index),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with exit status {return_code}; see the traceback above.')


def run_commands_parallel(jobs: list[tuple[list[str], str, int]]) -> None:
    processes: list[tuple[subprocess.Popen[str], str, int]] = []
    for command, label, gpu_index in jobs:
        resolved = resolve_cli_command(command, RUNTIME_PYTHON)
        print(f'Launching {label} on physical GPU {gpu_index}:', ' '.join(str(part) for part in resolved))
        process = subprocess.Popen(
            resolved,
            cwd=REPO_ROOT,
            env=pipeline_env(gpu_index),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        processes.append((process, label, gpu_index))

    def stream_output(process: subprocess.Popen[str], label: str, gpu_index: int) -> None:
        assert process.stdout is not None
        for line in process.stdout:
            print(f'[GPU {gpu_index} | {label}] {line}', end='')

    threads = [
        threading.Thread(target=stream_output, args=(process, label, gpu_index))
        for process, label, gpu_index in processes
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

    failures: list[str] = []
    for process, label, gpu_index in processes:
        if process.wait() != 0:
            failures.append(f'{label} on GPU {gpu_index}')
    if failures:
        raise RuntimeError('Parallel launch failed: ' + ', '.join(failures))


RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Config:', CONFIG_PATH)
print('Run root:', RUN_ROOT)
print('Physical GPU index:', PHYSICAL_GPU_INDEX)

## 5. Verify the GPU runtime

In [ ]:
print('All visible GPUs on this machine:')
list_physical_gpus()
print()
validate_gpu_index(PHYSICAL_GPU_INDEX)

## 5b. Optional parallel launcher

Use this only when you want to start multiple independent GPU jobs from one notebook. Each job must use a different physical GPU and a different output directory.

Example for four L4 cards:

```python
PARALLEL_JOBS = [
    (
        ['jspace-phase1', '--config', str(CONFIG_PATH), '--output-dir', str(REPO_ROOT / 'artifacts/jspace-qwen35_9b-smoke-gpu0/phase1'), '--stage', 'prepare'],
        'Phase 1 prepare GPU 0',
        0,
    ),
    (
        ['jspace-phase1', '--config', str(CONFIG_PATH), '--output-dir', str(REPO_ROOT / 'artifacts/jspace-qwen35_9b-smoke-gpu1/phase1'), '--stage', 'prepare'],
        'Phase 1 prepare GPU 1',
        1,
    ),
]
```

Leave `PARALLEL_JOBS` empty for the normal single-GPU workflow below.

In [ ]:
PARALLEL_JOBS: list[tuple[list[str], str, int]] = []

if PARALLEL_JOBS:
    run_commands_parallel(PARALLEL_JOBS)
else:
    print('PARALLEL_JOBS is empty. Continue with the single-GPU phase cells below.')

## 6. Run or resume Phase 1

This freezes the manifest, captures activations, reconstructs J-space, and selects the layer. Rerunning the cell reuses compatible caches.

In [ ]:
phase1_command = [
    'jspace-phase1',
    '--config', str(CONFIG_PATH),
    '--output-dir', str(PHASE1_DIR),
    '--stage', 'all',
]
if WEBQA_TRAIN_PATH is not None:
    phase1_command.extend(['--webqa-train', str(WEBQA_TRAIN_PATH)])
if SUMMARIZATION_TRAIN_PATH is not None:
    phase1_command.extend(['--summarization-train', str(SUMMARIZATION_TRAIN_PATH)])
run_command(phase1_command, 'Phase 1')

## 7. Inspect Phase 1 before continuing

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

selection = json.loads((PHASE1_DIR / 'selected_layer.json').read_text())
print(json.dumps(selection, indent=2))
display(pd.read_csv(PHASE1_DIR / 'layer_metrics.csv'))
display(Image(filename=str(PHASE1_DIR / 'layer_auprc.png')))
display(Image(filename=str(PHASE1_DIR / 'selected_layer_score_distribution.png')))

## 8. Run or resume Phase 2 generation

This GPU stage reads the frozen Phase 1 directory directly and runs three conditions: intact (`alpha=0.0`), partial removal (`alpha=0.5`), and full removal (`alpha=1.0`).

In [ ]:
phase2_base = [
    'jspace-phase2',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE2_DIR),
]
run_command([*phase2_base, '--stage', 'generate'], 'Phase 2 generation')

## 9. Run or resume Phase 2 analysis

This stage uses cached generations, ROUGE scoring, and the pinned OpenRouter judge.

In [ ]:
run_command([*phase2_base, '--stage', 'analyze'], 'Phase 2 analysis')

## 10. Inspect Phase 2 results

In [ ]:
display(pd.read_csv(PHASE2_DIR / 'phase2_summary.csv'))
display(pd.read_csv(PHASE2_DIR / 'phase2_examples.csv'))
display(Image(filename=str(PHASE2_DIR / 'phase2_asr_vs_alpha.png')))
display(Image(filename=str(PHASE2_DIR / 'phase2_clean_utility_vs_alpha.png')))

## 11. Construct and inspect the Phase 3 detectors

This CPU-only stage reads the frozen Phase 1 handoff directly. It does not load Gemma or the lens and does not depend on Phase 2.

In [ ]:
phase3_command = [
    'jspace-phase3',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE3_DIR),
]
run_command(phase3_command, 'Phase 3')
display(pd.read_csv(PHASE3_DIR / 'phase3_metrics.csv'))
display(Image(filename=str(PHASE3_DIR / 'phase3_detector_comparison.png')))

## 12. Run or resume and inspect Phase 4

This runs the three frozen transfer benchmarks. Generation requires CUDA; analysis uses CPU and OpenRouter only for BIPIA semantic outcomes.

In [ ]:
phase4_base = [
    'jspace-phase4',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--phase3', str(PHASE3_DIR),
    '--bipia-root', str(BIPIA_ROOT),
    '--agentdojo-root', str(AGENTDOJO_CHECKOUT),
    '--injecagent-root', str(INJECAGENT_CHECKOUT),
    '--output-dir', str(PHASE4_DIR),
]
run_command([*phase4_base, '--stage', 'generate'], 'Phase 4 generation')
run_command([*phase4_base, '--stage', 'analyze'], 'Phase 4 analysis')
display(pd.read_csv(PHASE4_DIR / 'phase4_metrics.csv'))
display(Image(filename=str(PHASE4_DIR / 'phase4_detector_transfer.png')))

## 13. Confirm persistence

All caches and results are written under the local run root. Preserve the `phase1/`, `phase2/`, `phase3/`, and `phase4/` directories together when copying or archiving a run.

In [ ]:
print('Complete run root:', RUN_ROOT)
print('Phase 1 selected layer:', selection['selected_layer'])
print('Phase 2 results:', PHASE2_DIR / 'phase2_results.parquet')
print('Phase 3 metrics:', PHASE3_DIR / 'phase3_metrics.csv')
print('Phase 4 metrics:', PHASE4_DIR / 'phase4_metrics.csv')
assert (PHASE1_DIR / 'selected_layer.json').is_file()
assert (PHASE2_DIR / 'phase2_results.parquet').is_file()
assert (PHASE3_DIR / 'mean_detector.pt').is_file()
assert (PHASE3_DIR / 'logistic_detector.pt').is_file()
assert (PHASE4_DIR / 'phase4_predictions.parquet').is_file()

## Interpretation boundary

An effect in Phase 2 shows that the selected layer's reconstructed J-space component is functionally involved in model behavior. Phase 3 measures development-set detectability and freezes thresholds. Phase 4 evaluates those frozen detectors on held-out and transfer benchmarks without tuning. None of these phases establishes injection-specific causality.